# IZQAR IEEE-CIS v4b — log1p causal frequency ablation (GPU CV)

Research mode only: final 15% physically excluded; 3 expanding chronological folds.
Arms `none` and `causal` were measured in v1; this version adds `causal_log` (log1p of prior counts).
tail is never used. No production artifact is exported.

In [ ]:
from pathlib import Path
import json, subprocess, sys

EXPECTED_REVISION = '5a8b8a41d1e34de470e572c608bf3410e9ef9452'
root = Path('/kaggle/working/izqar_ieee_v4b')
if root.exists():
    raise FileExistsError('Use fresh session')
subprocess.run(['git', 'clone', '--branch', 'feat/ieee-cis-research', 'https://github.com/Night011/izqar.git', str(root)], check=True)
subprocess.run(['git', 'checkout', '--detach', EXPECTED_REVISION], cwd=root, check=True)
actual = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=root, text=True).strip()
assert actual == EXPECTED_REVISION, f'unexpected revision {actual}'
backend = root / 'backend'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-q', '-r', str(backend / 'requirements-ml.txt')], check=True)
matches = list(Path('/kaggle/input').rglob('train_transaction.csv'))
assert len(matches) == 1, f'expected one train_transaction.csv, got {matches}'
data_dir = matches[0].parent
assert (data_dir / 'train_identity.csv').is_file(), 'train_identity.csv missing'
print('input:', data_dir, 'bytes:', matches[0].stat().st_size, 'revision:', actual)
import xgboost; print('xgboost', xgboost.__version__)

In [ ]:
import subprocess, sys, json
from pathlib import Path

arm = 'causal_log'
out = Path(f'/kaggle/working/ieee_v4_{arm}.joblib')
subprocess.run([sys.executable, str(backend / 'scripts' / 'train_ieee_v3.py'),
                '--data-dir', str(data_dir), '--output', str(out),
                '--mode', 'research', '--freq', arm, '--device', 'cuda',
                '--estimators', '1500'], cwd=backend, check=True)
report_path = out.with_suffix('.cv.metrics.json')
arm_report = json.loads(report_path.read_text(encoding='utf-8'))
print('== causal_log ==', json.dumps(arm_report['mean_metrics'], indent=2))
Path('/kaggle/working/ieee_v4_causal_log.cv.metrics.json').write_text(report_path.read_text(encoding='utf-8'), encoding='utf-8')
print('written /kaggle/working/ieee_v4_causal_log.cv.metrics.json')